# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will examine the Croissant schema to list record sets and their contained fields and columns using their `@id` values.

In [ ]:
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"RecordSet {i+1} @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print("  Fields and Columns:")
    for field in record_set.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, Data type: {field.data_type}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column @id: {col.id}, Name: {col.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

First, identify record set(s) to extract. Here, we will extract data for all record sets and demonstrate with the first.

In [ ]:
# Get the list of record set `@id`s
record_set_ids = [rs.id for rs in dataset.record_sets()]
print(f"Record set IDs: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows with columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print("No records found.")

# For the rest of the notebook, we'll use the first record set (if available)
if len(dataframes) > 0:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nUsing record set: {main_record_set_id} for further analysis.")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate basic EDA using one numeric field (identified by its `@id`) from the selected record set.

In [ ]:
import numpy as np
if main_record_set_id is not None and len(df) > 0:
    # List candidate numeric fields with their @id
    record_set = [rs for rs in dataset.record_sets() if rs.id == main_record_set_id][0]
    numeric_fields = []
    for field in record_set.fields:
        # Simple heuristic: check if data_type looks like float or int
        if hasattr(field, 'data_type') and ('Float' in str(field.data_type) or 'Integer' in str(field.data_type) or 'Number' in str(field.data_type)):
            if field.id in df.columns:
                numeric_fields.append((field.id, field.name))
    if len(numeric_fields) == 0:
        print("No numeric fields available in this record set.")
    else:
        print("Numeric fields available (by @id and name):")
        for f in numeric_fields:
            print(f)
        # Use the first numeric field
        numeric_field_id, numeric_field_name = numeric_fields[0]
        print(f"\nUsing numeric field: {numeric_field_id} ({numeric_field_name}) for filtering and normalization.")
        # Clean up data: Try converting to numeric
        numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = numeric_series.mean()  # Example: use mean as threshold
        filtered_df = df[numeric_series > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df[[numeric_field_id]].head())
        filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[numeric_series > threshold] - numeric_series.mean()) / numeric_series.std()
        print(f"Normalized {numeric_field_id} for filtered records (mean=0, std=1):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a categorical field if present
        group_field = None
        for field in record_set.fields:
            if hasattr(field, 'data_type') and ('Text' in str(field.data_type) or 'String' in str(field.data_type)):
                if field.id != numeric_field_id and field.id in df.columns:
                    group_field = field.id
                    break
        if group_field:
            print(f"\nGrouping data by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df[[f"{numeric_field_id}_normalized"]].head())
        else:
            print("No suitable group-by field found.")
else:
    print("No record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and len(df) > 0 and 'numeric_field_id' in locals():
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8,5))
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    sns.histplot(numeric_series.dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_name} ({numeric_field_id})')
    plt.show()
    # Boxplot grouped by category if available
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=numeric_series)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_name} by {group_field}')
        plt.show()
else:
    print("No visualization - no numeric field or data available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using its Croissant schema via `mlcroissant`, listed its record sets and fields by their `@id`, loaded records to pandas DataFrames, and performed simple exploratory data analysis and visualizations on numeric data fields using their `@id` identifiers.

You can extend this notebook by selecting additional fields or record sets, engineering new features, or conducting advanced analyses.

_Note: All references to record sets, fields, and columns in code are by their `@id`, as per Croissant best practices and for reproducibility across schema versions._